In [2]:
import numpy as np
import pandas as pd

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import SGDRegressor

import matplotlib.pyplot as plt

## Categorical preprocessing step

This code preprocesses the categorical (country) player information into a form that can be used in the learning algorithm.

We have a total of >130 nationalities, so quite a few.

![nationalities](nationalities.png)

The [documentation](https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features) lists two possibilities: the `Ordinal` and `OneHot` encoders. We could argue that onehot makes more sense, but it will introduce a lot (~130) of columns given our limited dataset (~5000 rows). The code below uses Ordinal transformations.

### Load Data (again)

In [29]:
data = pd.read_csv('../football_wages.csv')
data.head()

,age,height_cm,weight_kg,nationality_name,overall,potential,attacking_crossing,attacking_finishing,attacking_heading_accuracy,attacking_short_passing,...,movement_reactions,movement_balance,defending_standing_tackle,defending_sliding_tackle,goalkeeping_diving,goalkeeping_handling,goalkeeping_kicking,goalkeeping_positioning,goalkeeping_reflexes,log_wages
0,27.0,183.0,76.0,b'Korea Republic',57.0,58.0,54.0,30.0,55.0,53.0,...,60.0,67.0,63.0,58.0,9.0,13.0,8.0,11.0,10.0,3.000000
1,21.0,182.0,70.0,b'France',61.0,72.0,58.0,63.0,46.0,62.0,...,47.0,65.0,31.0,33.0,9.0,11.0,9.0,12.0,11.0,3.000000
2,35.0,182.0,75.0,b'Korea Republic',68.0,68.0,62.0,68.0,68.0,70.0,...,61.0,69.0,36.0,40.0,8.0,12.0,7.0,12.0,6.0,3.301030
3,29.0,169.0,70.0,b'Paraguay',67.0,67.0,62.0,55.0,50.0,71.0,...,59.0,84.0,40.0,55.0,6.0,10.0,11.0,15.0,9.0,2.698970
4,30.0,176.0,74.0,b'Austria',65.0,65.0,63.0,49.0,53.0,63.0,...,58.0,75.0,65.0,64.0,12.0,15.0,10.0,8.0,10.0,3.477121


### The Pipeline

In [93]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn import preprocessing

From the docs it seems that we can create a pipeline of preprocessing-to-classifier (https://scikit-learn.org/stable/modules/compose.html#build-a-pipeline).

Docs example for SVC:

```python
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.decomposition import PCA
estimators = [('reduce_dim', PCA()), ('clf', SVC())]
pipe = Pipeline(estimators)
pipe
```

### Categorical Preprocessing: no pipeline example

In [31]:
# Get sample player and see
player = data.iloc[42]
player

age                                 22.0
height_cm                          177.0
weight_kg                           70.0
nationality_name              b'Austria'
overall                             59.0
potential                           65.0
attacking_crossing                  52.0
attacking_finishing                 33.0
attacking_heading_accuracy          45.0
attacking_short_passing             51.0
attacking_volleys                   28.0
skill_dribbling                     51.0
skill_curve                         33.0
skill_fk_accuracy                   35.0
skill_long_passing                  48.0
skill_ball_control                  40.0
movement_acceleration               70.0
movement_sprint_speed               67.0
movement_agility                    68.0
movement_reactions                  56.0
movement_balance                    72.0
defending_standing_tackle           64.0
defending_sliding_tackle            63.0
goalkeeping_diving                  14.0
goalkeeping_hand

In [32]:
# Create a new encoded column
from sklearn.preprocessing import OrdinalEncoder

enc = OrdinalEncoder()

data_enc = data.copy()
data_enc['nationality_code'] = enc.fit_transform(data[['nationality_name']])
data_enc.drop('nationality_name', axis=1, inplace=True)

In [33]:
player_enc = data_enc.iloc[42]
player_enc

age                            22.0
height_cm                     177.0
weight_kg                      70.0
overall                        59.0
potential                      65.0
attacking_crossing             52.0
attacking_finishing            33.0
attacking_heading_accuracy     45.0
attacking_short_passing        51.0
attacking_volleys              28.0
skill_dribbling                51.0
skill_curve                    33.0
skill_fk_accuracy              35.0
skill_long_passing             48.0
skill_ball_control             40.0
movement_acceleration          70.0
movement_sprint_speed          67.0
movement_agility               68.0
movement_reactions             56.0
movement_balance               72.0
defending_standing_tackle      64.0
defending_sliding_tackle       63.0
goalkeeping_diving             14.0
goalkeeping_handling           13.0
goalkeeping_kicking            14.0
goalkeeping_positioning        11.0
goalkeeping_reflexes           12.0
log_wages                   

In [43]:
# see what values changed for player 42
print(f'Added {set(player_enc).difference(set(player))}, Removed {set(player).difference(set(player_enc))}')

Added {7.0}, Removed {"b'Austria'"}


### Can also be done inside a Pipeline (example)

In [98]:
from sklearn.compose import ColumnTransformer

# pipeline-like composition
p = FeatureUnion([
    # 1. encode categorical data
    ('replace_categorical', ColumnTransformer([('categorical_countries' , OrdinalEncoder(), ['nationality_name'])])),

    # 2. scale numeric data
    ('scale_numerical', ColumnTransformer([('numerical_Features', preprocessing.StandardScaler(), data.drop('nationality_name', axis=1).columns )]))
])

# player 42 -> 'Austria' is now '7', and remaining features are scaled.
p.fit_transform(data)[42]



array([ 7.        , -0.67576366, -0.66222704, -0.72559826, -0.98665631,
       -1.00601083,  0.14604913, -0.63990038, -0.38106189, -0.52256753,
       -0.81196942, -0.23575231, -0.76845984, -0.4131436 , -0.31583204,
       -1.07752087,  0.36232936,  0.16216387,  0.31166277, -0.61393509,
        0.56708444,  0.7525082 ,  0.82453949, -0.14751469, -0.19788368,
       -0.13132099, -0.31530706, -0.25833312, -0.87861844])